# V5.0 DF Experiment — Parameter Fix + 10-Fold CV + 100 Epochs

**V5 fixes from V4 analysis:**
- **EDL kl_lambda**: 0.10 to **0.05** (Phase 2: kl=0.05 Acc=80.00%, kl=0.10 only 70.00%)
- **EDL kl_anneal_cap**: 1.0 to **0.5** (matches kl=0.05, V4 analysis confirmed)
- **EDL+ORCU lambda_orcu**: 0.01 to **0.005** (Phase 3: lambda=0.005 mean 76.67%, lambda=0.01 only 66.67%)
- **Epochs**: 75 to **100** (fuller KL annealing, ES auto-stops at peak)
- **CV**: 5-fold to **10-fold** (improve statistical power ~40%)
- **CV coverage**: 3 methods to **all 5 methods** (complete paper comparison)

**Experiment Structure (77 runs, ~23h on T4 x2):**
- Phase 1: 5 methods x 3 seeds x 100 epochs = 15 runs (~5h)
- Phase 2: EDL kl fine-sweep [0.03, 0.05, 0.07] x seed=42 = 3 runs (~0.8h)
- Phase 3: EDL+ORCU lambda fine-sweep [0.003, 0.005, 0.008] x 3 seeds = 9 runs (~2.5h)
- Phase 4: 10-fold CV x 5 methods = 50 folds (~15h)

**Success Criteria:**
- S: >= 82% Acc, >= 81% F1 — Paper-ready SOTA
- **A: >= 80% Acc, >= 79% F1 — Beats all competitors (V5 target)**
- B: >= 78% Acc, >= 77% F1 — Competitive

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

# Clone updated code from GitHub
!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed!"
print("Code cloned successfully.")


In [ ]:
# Master Setup -- cache pretrained weights while internet is available
import os, sys, json, copy, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

print("Downloading pretrained weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Weights cached.")

CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("Setup complete.")


## 2. Data Verification

In [ ]:
def verify_df():
    ds_cls = DFDataset
    for split in ["train", "val", "test"]:
        ds = ds_cls(DATA_ROOT, split=split)
        labels = np.array([ds[i][1] for i in range(len(ds))])
        dist = {f"G{k}": int((labels == k).sum()) for k in range(4)}
        print(f"  DF {split:5s}: {len(ds):3d} samples | {dist}")

print("=== Data Verification ===")
verify_df()
print("OK.")


## 3. V5 Training Components

In [ ]:
# ---- MixUp (for CE/Cumulative/SORD modes) ----
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("MixUp helpers ready.")


In [ ]:
# ---- EarlyStopping ----
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = -float("inf") if mode == "max" else float("inf")
        self.counter = 0
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, metric, epoch):
        if self.mode == "max":
            improved = metric > self.best + self.min_delta
        else:
            improved = metric < self.best - self.min_delta
        if improved:
            self.best = metric
            self.counter = 0
            self.best_epoch = epoch
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

print("EarlyStopping ready.")


In [ ]:
# ---- V5 Training Function (corrected params from V4 analysis) ----
@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets)


def train_model_v5(task, data_root, output_dir, mode="edl_orcu",
                   batch_size=32, epochs=100,
                   lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.05,
                   warmup_epochs=5,
                   kl_lambda=None, kl_anneal_cap=None,
                   lambda_orcu=0.005, lambda_reg=0.005, orcu_t=3.0,
                   stage_1_epochs=3, stage_2_epochs=10,
                   use_mixup=True, mixup_alpha=0.2,
                   patience=None, seed=42):
    """V5 training: corrected params from V4 analysis.

    V5 defaults (method-specific, FIXED from V4):
      - EDL:        kl_lambda=0.05, kl_anneal_cap=0.5, patience=25
      - EDL+ORCU:   kl_lambda=0.05, kl_anneal_cap=0.5, patience=15, lambda_orcu=0.005
      - CE/Cum/SORD: patience=15, MixUp active
      - ALL: epochs=100 (increased from 75)
    """

    # ---- Method-specific defaults (V5 corrected) ----
    if patience is None:
        patience = 25 if mode == "edl" else 15
    if kl_lambda is None:
        if mode == "edl":
            kl_lambda = 0.05       # V5 fix: 0.10 -> 0.05 (V4 Phase 2 best)
        elif mode == "edl_orcu":
            kl_lambda = 0.05       # V5 fix: kept at 0.05
        else:
            kl_lambda = 0.02       # Not used for CE/Cum/SORD
    if kl_anneal_cap is None:
        kl_anneal_cap = 0.5        # V5 fix: 1.0 -> 0.5 for EDL

    torch.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    model = build_model(task=task, mode=mode)
    model.to(device)
    model_type = type(model).__name__
    print(f"Model: {sum(p.numel() for p in model.parameters()):,} params | "
          f"Mode: {mode} | Type: {model_type}")

    mixup_modes = {"ce", "cumulative", "sord"}
    apply_mixup = use_mixup and mode in mixup_modes
    if apply_mixup:
        print(f"  MixUp: alpha={mixup_alpha} (active)")
    else:
        print(f"  MixUp: skipped (not applicable for {mode})")

    print(f"  V5 Params: epochs={epochs} patience={patience} seed={seed}")
    if mode in ("edl", "edl_orcu"):
        print(f"  KL: lambda={kl_lambda} anneal_cap={kl_anneal_cap}")
    if mode == "edl_orcu":
        print(f"  ORCU: lambda_orcu={lambda_orcu} lambda_reg={lambda_reg} "
              f"stages={stage_1_epochs}/{stage_2_epochs}")

    # ---- Criterion ----
    if mode == "ce":
        def criterion(a, z, t, e):
            loss = F.nll_loss(F.log_softmax(z, dim=-1), t)
            return loss, {"stage": 0, "L_ce": loss.item()}
    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cum_fn = CumulativeLoss(num_classes=4)
        def criterion(a, z, t, e):
            ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
            loss = cum_fn(ol, t)
            return loss, {"stage": 0, "L_cum": loss.item()}
    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sord_fn = ORCULoss(num_classes=4, t=orcu_t, lambda_reg=0.0)
        def criterion(a, z, t, e):
            loss = sord_fn(z, t)
            return loss, {"stage": 0, "L_sord": loss.item()}
    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        edl_fn = EDLLoss(num_classes=4, kl_lambda=kl_lambda, kl_anneal_cap=kl_anneal_cap)
        def criterion(a, z, t, e):
            loss = edl_fn(a, t, e, epochs)
            return loss, {"stage": 0, "L_edl": loss.item()}
    else:  # edl_orcu
        criterion = CombinedLoss(
            num_classes=4, lambda_orcu=lambda_orcu, lambda_kl=kl_lambda,
            kl_anneal_cap=kl_anneal_cap,
            orcu_t=orcu_t, orcu_lambda_reg=lambda_reg,
            stage_1_epochs=stage_1_epochs, stage_2_epochs=stage_2_epochs,
            total_epochs=epochs)

    # ---- Optimizer & Scheduler ----
    bb_params, hd_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb_params if n.startswith("backbone") else hd_params).append(p)
    optimizer = optim.AdamW([
        {"params": bb_params, "lr": lr_backbone},
        {"params": hd_params, "lr": lr_head},
    ], weight_decay=weight_decay)

    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched],
                             milestones=[warmup_epochs])

    # ---- Training Loop ----
    early_stopper = EarlyStopping(patience=patience, mode="max")
    best_val_acc, best_state, best_epoch = 0.0, None, 0
    stopped_early = False
    history = []

    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_cum": 0.0, "L_sord": 0.0,
                        "L_edl": 0.0, "L_orcu": 0.0, "L_total": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)

            if apply_mixup:
                images, targets_a, targets_b, lam = mixup_data(
                    images, targets, alpha=mixup_alpha)

            alpha_out, z = model(images)

            if apply_mixup:
                loss_a, loss_info = criterion(alpha_out, z, targets_a, epoch)
                loss_b, _ = criterion(alpha_out, z, targets_b, epoch)
                loss = lam * loss_a + (1 - lam) * loss_b
            else:
                loss, loss_info = criterion(alpha_out, z, targets, epoch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        val_metrics = evaluate_model(model, val_loader)
        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch

        history.append({
            "epoch": epoch, "stage": epoch_stage,
            "train_loss": epoch_losses,
            "val_metrics": {k: v for k, v in val_metrics.items()
                            if isinstance(v, (float, int)) and not k.startswith("evidence_class_")},
        })

        loss_key = next((k for k in ["L_total", "L_ce", "L_cum", "L_sord", "L_edl", "L_orcu"]
                         if epoch_losses.get(k, 0) != 0), "L_total")
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | "
                  f"{loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f} "
                  f"QWK={val_metrics['qwk']:.4f} ECE={val_metrics['ece']:.4f} "
                  f"ES={early_stopper.counter}/{patience}")

        if early_stopper(val_metrics["acc"], epoch):
            print(f"  EarlyStopping at epoch {epoch+1} "
                  f"(best: epoch {best_epoch+1}, val_acc={best_val_acc:.4f})")
            stopped_early = True
            break

    if not stopped_early:
        print(f"  Completed {epochs} epochs (best: epoch {best_epoch+1}, "
              f"val_acc={best_val_acc:.4f})")

    # Load best and test
    model.load_state_dict(best_state)
    test_metrics = evaluate_model(model, test_loader)

    # ---- Save ----
    torch.save({"model_state_dict": best_state, "test_metrics": test_metrics,
                "best_epoch": best_epoch, "stopped_early": stopped_early},
               os.path.join(output_dir, "best.pt"))

    serializable = {}
    for k, v in test_metrics.items():
        if isinstance(v, (float, int, bool)):
            serializable[k] = v
        elif isinstance(v, (np.floating, np.integer)):
            serializable[k] = float(v)
        elif isinstance(v, np.ndarray):
            serializable[k] = v.tolist()
        elif isinstance(v, list):
            serializable[k] = v
    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump(serializable, f, indent=2)
    with open(os.path.join(output_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Per-sample prediction export
    from eval import export_predictions, save_predictions
    pred_data = export_predictions(model, test_loader, device)
    npz_path = os.path.join(output_dir, f"v5_{mode}_seed{seed}_predictions.npz")
    save_predictions(pred_data, npz_path, mode=mode, task=task, seed=seed)

    print(f"\n[Test] Acc={test_metrics['acc']:.4f} "
          f"MacroF1={test_metrics['macro_f1']:.4f} "
          f"WeightedF1={test_metrics['weighted_f1']:.4f} "
          f"QWK={test_metrics['qwk']:.4f} ECE={test_metrics['ece']:.4f} "
          f"BestEp={best_epoch+1} EarlyStop={stopped_early}")
    if "per_class_f1" in test_metrics:
        pf1 = test_metrics["per_class_f1"]
        print(f"  Per-Class F1: N={pf1[0]:.4f} Mi={pf1[1]:.4f} "
              f"Mo={pf1[2]:.4f} Se={pf1[3]:.4f}")
    return test_metrics, history

print("train_model_v5() ready. V5 defaults: epochs=100 EDL(kl=0.05,cap=0.5) EDL+ORCU(lambda_orcu=0.005)")


## 4. Phase 1: Core Methods x Multi-Seed (15 runs)

**V5 defaults per method:**
| Method | kl_lambda | kl_cap | lambda_orcu | Patience | MixUp | Seeds |
|--------|-----------|--------|-------------|----------|-------|-------|
| CE | -- | -- | -- | 15 | alpha=0.2 | [42,123,456] |
| Cumulative | -- | -- | -- | 15 | alpha=0.2 | [42,123,456] |
| SORD | -- | -- | -- | 15 | alpha=0.2 | [42,123,456] |
| EDL | **0.05** | **0.5** | -- | **25** | No | [42,123,456] |
| EDL+ORCU | **0.05** | 0.5 | **0.005** | 15 | No | [42,123,456] |

**V4 -> V5:** EDL kl 0.10->0.05, EDL cap 1.0->0.5, EDL+ORCU lambda 0.01->0.005, epochs 75->100

Expected: ~5h on T4 x2

In [ ]:
print("=" * 70)
print("DF V5 Phase 1 -- Core Methods x Multi-Seed [42, 123, 456]")
print("=" * 70)
print("V5: EDL kl=0.05 cap=0.5 | EDL+ORCU kl=0.05 lambda_orcu=0.005 | epochs=100")

SEEDS = [42, 123, 456]
MODES = ["ce", "cumulative", "sord", "edl", "edl_orcu"]
EPOCHS = 100

phase1_results = {}

for mode in MODES:
    phase1_results[mode] = {}
    for seed in SEEDS:
        print(f"\n{'='*60}")
        print(f"DF V5 Phase 1 | Mode: {mode} | Seed: {seed}")
        print(f"{'='*60}")
        out_dir = os.path.join(OUTPUT_DIR, f"v5_p1_{mode}_s{seed}")
        metrics, history = train_model_v5(
            "df", DATA_ROOT, out_dir, mode=mode,
            epochs=EPOCHS, seed=seed)
        phase1_results[mode][str(seed)] = {
            k: v for k, v in metrics.items()
            if isinstance(v, (float, int, bool, list))
        }

with open(os.path.join(OUTPUT_DIR, "v5_phase1_results.json"), "w") as f:
    json.dump(phase1_results, f, indent=2)

print("\n" + "=" * 100)
print("PHASE 1 RESULTS -- Multi-Seed Summary")
print("=" * 100)

print(f"\n{'Mode':<14} {'Seed':>6} {'Acc':>8} {'MacroF1':>8} {'WgtF1':>8} "
      f"{'QWK':>8} {'ECE':>8} {'%Unim':>8}")
print("-" * 84)
for mode in MODES:
    for seed_str in [str(s) for s in SEEDS]:
        m = phase1_results[mode].get(seed_str, {})
        print(f"{mode:<14} {seed_str:>6} "
              f"{m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
              f"{m.get('weighted_f1',0):8.4f} {m.get('qwk',0):8.4f} "
              f"{m.get('ece',0):8.4f} {m.get('pct_unimodal',0):7.2%}")

print(f"\n{'Mode':<14} {'Acc':>16} {'MacroF1':>16} {'WgtF1':>16} {'QWK':>16}")
print("-" * 82)
for mode in MODES:
    accs = [phase1_results[mode][str(s)]['acc'] for s in SEEDS]
    f1s = [phase1_results[mode][str(s)].get('macro_f1',0) for s in SEEDS]
    wf1s = [phase1_results[mode][str(s)].get('weighted_f1',0) for s in SEEDS]
    qwks = [phase1_results[mode][str(s)].get('qwk',0) for s in SEEDS]
    print(f"{mode:<14} {np.mean(accs):.4f}+/-{np.std(accs):.4f}  "
          f"{np.mean(f1s):.4f}+/-{np.std(f1s):.4f}  "
          f"{np.mean(wf1s):.4f}+/-{np.std(wf1s):.4f}  "
          f"{np.mean(qwks):.4f}+/-{np.std(qwks):.4f}")

print(f"\n{'Mode':<14} {'Normal':>10} {'Mild':>10} {'Moderate':>10} {'Severe':>10}")
print("-" * 60)
for mode in MODES:
    pf1s = []
    for s in SEEDS:
        pf1 = phase1_results[mode][str(s)].get('per_class_f1', [0,0,0,0])
        pf1s.append(pf1)
    avg_pf1 = np.mean(pf1s, axis=0)
    print(f"{mode:<14} {avg_pf1[0]:10.4f} {avg_pf1[1]:10.4f} "
          f"{avg_pf1[2]:10.4f} {avg_pf1[3]:10.4f}")

print("\nPhase 1 complete. v5_phase1_results.json saved.")


## 5. Phase 2: EDL kl Fine-Sweep (3 runs, seed=42)

V5 fine-sweeps around kl=0.05 (V4's best):

| kl_lambda | kl_cap | Purpose |
|-----------|--------|---------|
| 0.03 | 0.5 | Lower bound check |
| **0.05** | **0.5** | V5 default |
| 0.07 | 0.5 | Upper bound check |

Expected: ~0.8h on T4 x2

In [ ]:
print("=" * 70)
print("DF V5 Phase 2 -- EDL kl Fine-Sweep")
print("=" * 70)

KL_VALUES = [0.03, 0.05, 0.07]
KL_CAPS   = [0.5,  0.5,  0.5]
SWEEP_SEED = 42

phase2_results = []

for kl_val, kl_cap in zip(KL_VALUES, KL_CAPS):
    print(f"\n{'='*50}")
    print(f"EDL kl_lambda={kl_val}, kl_cap={kl_cap} | seed={SWEEP_SEED} | epochs=100")
    print(f"{'='*50}")
    out_dir = os.path.join(OUTPUT_DIR, f"v5_p2_edl_kl{kl_val:.2f}_cap{kl_cap}")
    metrics, history = train_model_v5(
        "df", DATA_ROOT, out_dir, mode="edl",
        kl_lambda=kl_val, kl_anneal_cap=kl_cap,
        epochs=EPOCHS, seed=SWEEP_SEED)

    result = {
        "kl_lambda": kl_val, "kl_anneal_cap": kl_cap, "seed": SWEEP_SEED,
        "acc": metrics["acc"], "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "qwk": metrics["qwk"], "ece": metrics["ece"],
        "pct_unimodal": metrics["pct_unimodal"],
        "u_ece": metrics["u_ece"], "auroc_u": metrics["auroc_u"],
    }
    if "per_class_f1" in metrics:
        result["per_class_f1"] = metrics["per_class_f1"]
    phase2_results.append(result)

with open(os.path.join(OUTPUT_DIR, "v5_phase2_kl_sweep.json"), "w") as f:
    json.dump(phase2_results, f, indent=2)

print(f"\n{'='*90}")
print("Phase 2 Results -- EDL kl Fine-Sweep")
print(f"{'='*90}")
hdr = f"{'kl':>8} {'cap':>6} {'Acc':>8} {'MacroF1':>8} {'WgtF1':>8} {'QWK':>8} {'ECE':>8} {'%Unim':>8}"
print(hdr)
print("-" * 80)
for r in sorted(phase2_results, key=lambda x: x["acc"], reverse=True):
    print(f"{r['kl_lambda']:8.2f} {r['kl_anneal_cap']:6.1f} "
          f"{r['acc']:8.4f} {r['macro_f1']:8.4f} {r['weighted_f1']:8.4f} "
          f"{r['qwk']:8.4f} {r['ece']:8.4f} {r['pct_unimodal']:7.2%}")

best_kl = max(phase2_results, key=lambda x: x["acc"])
print(f"\nBest kl: {best_kl['kl_lambda']} (Acc={best_kl['acc']:.4f}, F1={best_kl['macro_f1']:.4f})")
print("\nPhase 2 complete. v5_phase2_kl_sweep.json saved.")


## 6. Phase 3: EDL+ORCU lambda_orcu Fine-Sweep (9 runs)

V5 fine-sweeps around lambda=0.005 (V4's best):

| lambda_orcu | Seeds | Purpose |
|-------------|-------|---------|
| 0.003 | [42,123,456] | Weaker ORCU constraint |
| 0.005 | [42,123,456] | V5 default |
| 0.008 | [42,123,456] | Stronger ORCU constraint |

Expected: ~2.5h on T4 x2

In [ ]:
print("=" * 70)
print("DF V5 Phase 3 -- EDL+ORCU lambda_orcu Fine-Sweep")
print("=" * 70)

LAMBDA_ORCU_VALS = [0.003, 0.005, 0.008]
ORCU_SEEDS = [42, 123, 456]

phase3_results = []

for lam_orcu in LAMBDA_ORCU_VALS:
    for seed in ORCU_SEEDS:
        print(f"\n{'='*50}")
        print(f"EDL+ORCU lambda_orcu={lam_orcu} | seed={seed} | epochs=100")
        print(f"{'='*50}")
        out_dir = os.path.join(OUTPUT_DIR, f"v5_p3_orcu_lam{lam_orcu}_s{seed}")
        metrics, history = train_model_v5(
            "df", DATA_ROOT, out_dir, mode="edl_orcu",
            lambda_orcu=lam_orcu, epochs=EPOCHS, seed=seed)

        result = {
            "lambda_orcu": lam_orcu, "seed": seed,
            "acc": metrics["acc"], "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "qwk": metrics["qwk"], "ece": metrics["ece"],
            "pct_unimodal": metrics["pct_unimodal"],
            "u_ece": metrics["u_ece"], "auroc_u": metrics["auroc_u"],
        }
        if "per_class_f1" in metrics:
            result["per_class_f1"] = metrics["per_class_f1"]
        phase3_results.append(result)

with open(os.path.join(OUTPUT_DIR, "v5_phase3_orcu_sweep.json"), "w") as f:
    json.dump(phase3_results, f, indent=2)

# Summary
print(f"\n{'lambda_orcu':>8} {'Seed':>6} {'Acc':>8} {'MacroF1':>8} {'WgtF1':>8} "
      f"{'QWK':>8} {'ECE':>8} {'%Unim':>8}")
print("-" * 82)
for r in sorted(phase3_results, key=lambda x: (x["lambda_orcu"], x["seed"])):
    print(f"{r['lambda_orcu']:8.4f} {r['seed']:>6} "
          f"{r['acc']:8.4f} {r['macro_f1']:8.4f} {r['weighted_f1']:8.4f} "
          f"{r['qwk']:8.4f} {r['ece']:8.4f} {r['pct_unimodal']:7.2%}")

print(f"\n{'lambda_orcu':>8} {'Acc':>16} {'MacroF1':>16} {'WgtF1':>16} {'QWK':>16}")
print("-" * 82)
for lam in LAMBDA_ORCU_VALS:
    subset = [r for r in phase3_results if r["lambda_orcu"] == lam]
    accs = [r["acc"] for r in subset]
    f1s = [r["macro_f1"] for r in subset]
    wf1s = [r["weighted_f1"] for r in subset]
    qwks = [r["qwk"] for r in subset]
    print(f"{lam:8.4f} {np.mean(accs):.4f}+/-{np.std(accs):.4f}  "
          f"{np.mean(f1s):.4f}+/-{np.std(f1s):.4f}  "
          f"{np.mean(wf1s):.4f}+/-{np.std(wf1s):.4f}  "
          f"{np.mean(qwks):.4f}+/-{np.std(qwks):.4f}")

best = max(phase3_results, key=lambda x: x["acc"])
lam_means = {}
for lam in LAMBDA_ORCU_VALS:
    subset = [r for r in phase3_results if r["lambda_orcu"] == lam]
    lam_means[lam] = np.mean([r["acc"] for r in subset])
best_lam = max(lam_means, key=lam_means.get)

print(f"\nBest single: lambda={best['lambda_orcu']} s{best['seed']} Acc={best['acc']:.4f}")
print(f"Best lambda by mean: {best_lam} (mean Acc={lam_means[best_lam]:.4f})")
print("\nPhase 3 complete. v5_phase3_orcu_sweep.json saved.")


## 7. Phase 4: 10-Fold CV — All 5 Methods (50 folds)

**V5 changes from V4:** 10-fold (not 5), all 5 methods (not 3), V5-corrected params.

| Method | Config |
|--------|--------|
| CE | MixUp alpha=0.2, patience=15 |
| Cumulative | MixUp alpha=0.2, patience=15 |
| SORD | MixUp alpha=0.2, patience=15 |
| **EDL** | **kl_lambda=0.05, kl_cap=0.5, patience=25** |
| **EDL+ORCU** | **kl_lambda=0.05, lambda_orcu=0.005, kl_cap=0.5, patience=15** |

Expected: ~15h on T4 x2

In [ ]:
# ---- 10-Fold CV (V5 params, all 5 methods) ----
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

def run_cv_v5(data_root, methods_config, n_folds=10, epochs=100, seed=42):
    """V5 CV: 10-fold, all methods with V5-corrected defaults."""

    label_ds = DFDataset(data_root, split="train", split_seed=seed)
    labels = np.array([label_ds[i][1] for i in range(len(label_ds))])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    method_names = [m[0] for m in methods_config]
    fold_results = {mn: [] for mn in method_names}

    for fold_idx, (train_idx, val_idx) in enumerate(
            skf.split(np.zeros(len(label_ds)), labels)):
        print(f"\n{'='*50}")
        print(f"DF CV V5 Fold {fold_idx+1}/{n_folds}")
        print(f"{'='*50}")

        train_full = DFDataset(data_root, split="train", split_seed=seed)
        val_full = DFDataset(data_root, split="train", split_seed=seed)
        train_full.transform = get_transforms("df", is_train=True)
        val_full.transform = get_transforms("df", is_train=False)

        train_subset = Subset(train_full, train_idx)
        val_subset = Subset(val_full, val_idx)

        test_ds = DFDataset(data_root, split="test")
        test_ds.transform = get_transforms("df", is_train=False)

        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False,
                                num_workers=2, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                                 num_workers=2, pin_memory=True)

        for method_name, kwargs in methods_config:
            print(f"  [{method_name}] training...")
            fold_seed = seed + fold_idx * 100
            torch.manual_seed(fold_seed)
            np.random.seed(fold_seed)
            model = build_model(task="df", mode=method_name)
            model.to(device)

            kl_lambda = kwargs.get("kl_lambda", 0.02)
            kl_cap = kwargs.get("kl_anneal_cap", 0.5)

            if method_name == "ce":
                def crit(a, z, t, e):
                    loss = F.nll_loss(F.log_softmax(z, dim=-1), t)
                    return loss, {"stage": 0, "L_ce": loss.item()}
            elif method_name == "cumulative":
                from losses.cumulative_loss import CumulativeLoss
                cf = CumulativeLoss(num_classes=4)
                def crit(a, z, t, e):
                    ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                    loss = cf(ol, t)
                    return loss, {"stage": 0, "L_cum": loss.item()}
            elif method_name == "sord":
                from losses.orcu_loss import ORCULoss
                sf = ORCULoss(num_classes=4, t=3.0, lambda_reg=0.0)
                def crit(a, z, t, e):
                    loss = sf(z, t)
                    return loss, {"stage": 0, "L_sord": loss.item()}
            elif method_name == "edl":
                from losses.edl_loss import EDLLoss
                ef = EDLLoss(num_classes=4, kl_lambda=kl_lambda, kl_anneal_cap=kl_cap)
                def crit(a, z, t, e):
                    loss = ef(a, t, e, epochs)
                    return loss, {"stage": 0, "L_edl": loss.item()}
            else:  # edl_orcu
                crit = CombinedLoss(
                    num_classes=4,
                    lambda_orcu=kwargs.get("lambda_orcu", 0.005),
                    lambda_kl=kl_lambda, kl_anneal_cap=kl_cap,
                    orcu_t=3.0, orcu_lambda_reg=kwargs.get("lambda_reg", 0.005),
                    stage_1_epochs=3, stage_2_epochs=10, total_epochs=epochs)

            bb, hd = [], []
            for n, p in model.named_parameters():
                if not p.requires_grad:
                    continue
                (bb if n.startswith("backbone") else hd).append(p)
            optimizer = optim.AdamW(
                [{"params": bb, "lr": 1e-4}, {"params": hd, "lr": 1e-3}],
                weight_decay=0.05)
            ws = LinearLR(optimizer, start_factor=0.1, total_iters=5)
            cs = CosineAnnealingLR(optimizer, T_max=epochs - 5)
            scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[5])

            patience = kwargs.get("patience", 15)
            es = EarlyStopping(patience=patience, mode="max")
            best_val_acc, best_state = 0.0, None

            @torch.no_grad()
            def eval_cv(m, loader):
                m.eval()
                aa, zz, tt = [], [], []
                for im, tg in loader:
                    im, tg = im.to(device), tg.to(device)
                    a, z_ = m(im)
                    if a is not None:
                        aa.append(a.cpu())
                    zz.append(z_.cpu())
                    tt.append(tg.cpu())
                a = torch.cat(aa) if aa else None
                return compute_metrics(a, torch.cat(zz), torch.cat(tt))

            for epoch in range(epochs):
                model.train()
                for images, targets in train_loader:
                    images, targets = images.to(device), targets.to(device)
                    alpha, z = model(images)
                    loss, _ = crit(alpha, z, targets, epoch)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                scheduler.step()

                vm = eval_cv(model, val_loader)
                if vm["acc"] > best_val_acc:
                    best_val_acc = vm["acc"]
                    best_state = copy.deepcopy(model.state_dict())
                if es(vm["acc"], epoch):
                    break

            model.load_state_dict(best_state)
            tm = eval_cv(model, test_loader)
            fold_results[method_name].append({
                "fold": fold_idx,
                **{k: v for k, v in tm.items()
                   if isinstance(v, (float, int, bool, list, np.ndarray))}
            })
            print(f"    Acc={tm['acc']:.4f} MacroF1={tm['macro_f1']:.4f} "
                  f"WgtF1={tm['weighted_f1']:.4f} QWK={tm['qwk']:.4f}")

    # Aggregate
    summary = {}
    metric_names = ["acc", "macro_f1", "weighted_f1", "qwk", "ece",
                    "pct_unimodal", "u_ece", "auroc_u"]
    for method_name in method_names:
        agg = {}
        for mn in metric_names:
            vals = [f[mn] for f in fold_results[method_name] if mn in f]
            if vals:
                agg[mn] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)),
                           "values": [float(v) if not isinstance(v, list) else v for v in vals]}
        summary[method_name] = agg

    for method_name in method_names:
        if "per_class_f1" in fold_results[method_name][0]:
            pf1s = np.array([f["per_class_f1"] for f in fold_results[method_name]])
            summary[method_name]["per_class_f1"] = {
                "mean": pf1s.mean(axis=0).tolist(),
                "std": pf1s.std(axis=0).tolist()
            }

    # Paired t-tests (all pairs)
    significance = {}
    for i, m1 in enumerate(method_names):
        for j, m2 in enumerate(method_names):
            if i >= j:
                continue
            pair_key = f"{m1}_vs_{m2}"
            significance[pair_key] = {}
            for mn in ["acc", "macro_f1", "weighted_f1", "qwk", "ece"]:
                v1 = [f[mn] for f in fold_results[m1]]
                v2 = [f[mn] for f in fold_results[m2]]
                t_stat, p_val = stats.ttest_rel(v1, v2)
                significance[pair_key][mn] = {
                    "t_statistic": float(t_stat), "p_value": float(p_val),
                    "significant": bool(p_val < 0.05)}

    return {"task": "df", "k": n_folds, "methods": method_names,
            "fold_results": fold_results, "summary": summary,
            "significance": significance}

print("run_cv_v5() ready.")

# ---- Run Phase 4 ----
print("\\n" + "#" * 60)
print("# DF V5 Phase 4: 10-Fold CV -- All 5 Methods")
print("#" * 60)

cv_configs = [
    ("ce", {"patience": 15}),
    ("cumulative", {"patience": 15}),
    ("sord", {"patience": 15}),
    ("edl", {"kl_lambda": 0.05, "kl_anneal_cap": 0.5, "patience": 25}),
    ("edl_orcu", {"kl_lambda": 0.05, "lambda_orcu": 0.005,
                  "lambda_reg": 0.005, "kl_anneal_cap": 0.5, "patience": 15}),
]

cv_result = run_cv_v5(DATA_ROOT, methods_config=cv_configs,
                      n_folds=10, epochs=EPOCHS)

def convert(obj):
    if isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert(v) for v in obj]
    elif isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(os.path.join(OUTPUT_DIR, "v5_phase4_cv.json"), "w") as f:
    json.dump(convert(cv_result), f, indent=2)

print("\\n=== Phase 4 CV Summary ===")
print(f"{'Method':<14} {'Acc':>18} {'MacroF1':>18} {'WgtF1':>18} {'QWK':>18} {'ECE':>18}")
print("-" * 98)
for method, agg in cv_result["summary"].items():
    print(f"{method:<14} "
          f"{agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
          f"{agg['macro_f1']['mean']:.4f}+/-{agg['macro_f1']['std']:.4f}  "
          f"{agg['weighted_f1']['mean']:.4f}+/-{agg['weighted_f1']['std']:.4f}  "
          f"{agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}  "
          f"{agg['ece']['mean']:.4f}+/-{agg['ece']['std']:.4f}")

print(f"\\n{'Method':<14} {'Normal':>12} {'Mild':>12} {'Moderate':>12} {'Severe':>12}")
print("-" * 68)
for method, agg in cv_result["summary"].items():
    if "per_class_f1" in agg:
        pf1_mean = agg["per_class_f1"]["mean"]
        pf1_std = agg["per_class_f1"]["std"]
        print(f"{method:<14} "
              f"{pf1_mean[0]:.4f}+/-{pf1_std[0]:.4f}  "
              f"{pf1_mean[1]:.4f}+/-{pf1_std[1]:.4f}  "
              f"{pf1_mean[2]:.4f}+/-{pf1_std[2]:.4f}  "
              f"{pf1_mean[3]:.4f}+/-{pf1_std[3]:.4f}")

print("\\n  Significance (p < 0.05):")
for pair, tests in cv_result["significance"].items():
    sigs = [f"{m}(p={v['p_value']:.3f})" for m, v in tests.items() if v['significant']]
    print(f"    {pair}: {', '.join(sigs) if sigs else 'none'}")

print("\\nPhase 4 complete. v5_phase4_cv.json saved.")


## 8. V5 Final Results Summary

In [ ]:
print("\n" + "=" * 90)
print("V5 FINAL RESULTS SUMMARY")
print("=" * 90)

all_summary = {}
for fname, key in [("v5_phase1_results.json", "phase1"),
                   ("v5_phase2_kl_sweep.json", "phase2"),
                   ("v5_phase3_orcu_sweep.json", "phase3"),
                   ("v5_phase4_cv.json", "phase4")]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        with open(fpath) as f:
            all_summary[key] = json.load(f)

# ---- Phase 1 ----
print("\n--- Phase 1: Core Methods x Multi-Seed ---")
if "phase1" in all_summary:
    p1 = all_summary["phase1"]
    SEEDS_V = [42, 123, 456]
    MODES_V = ["ce", "cumulative", "sord", "edl", "edl_orcu"]
    for mode in MODES_V:
        if mode in p1:
            accs = [p1[mode][str(s)]['acc'] for s in SEEDS_V]
            f1s = [p1[mode][str(s)].get('macro_f1',0) for s in SEEDS_V]
            print(f"  {mode:<14}: Acc={np.mean(accs):.4f}+/-{np.std(accs):.4f}  "
                  f"F1={np.mean(f1s):.4f}+/-{np.std(f1s):.4f}")

# ---- Phase 4 ----
print("\n--- Phase 4: 10-Fold CV -- All 5 Methods ---")
if "phase4" in all_summary:
    cv_summary = all_summary["phase4"].get("summary", {})
    for method, agg in cv_summary.items():
        if isinstance(agg, dict) and "acc" in agg:
            print(f"  {method:<14}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
                  f"F1={agg['macro_f1']['mean']:.4f}+/-{agg['macro_f1']['std']:.4f}  "
                  f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}")

# ---- Competitors ----
print("\n" + "=" * 90)
print("V5 vs COMPETITORS")
print("=" * 90)
print(f"\n{'Method':<20} {'Acc':>10} {'MacroF1':>10} {'Params':>10} {'Uncertainty':>14}")
print("-" * 64)
print(f"{'MLTrMR (2025)':<20} {'80.19%':>10} {'75.79%':>10} {'556M':>10} {'None':>14}")
print(f"{'LD2Net (2026)':<20} {'80.00%':>10} {'79.88%':>10} {'3.3M':>10} {'None':>14}")
print(f"{'FusionDentNet (2024)':<20} {'80.00%':>10} {'79.25%':>10} {'201M':>10} {'None':>14}")
print(f"{'HiFuse (2024)':<20} {'78.23%':>10} {'70.45%':>10} {'164M':>10} {'None':>14}")
print(f"{'--'*64}")
if "phase4" in all_summary:
    for method, agg in cv_summary.items():
        if isinstance(agg, dict) and "acc" in agg:
            unc = "Dirichlet" if method in ("edl", "edl_orcu") else "Entropy"
            print(f"{'Ours '+method.upper()+' (V5)':<20} "
                  f"{agg['acc']['mean']:.2%}+/-{agg['acc']['std']:.2%}  "
                  f"{agg['macro_f1']['mean']:.2%}+/-{agg['macro_f1']['std']:.2%}  "
                  f"{'86M':>10} {unc:>14}")

# ---- Tier ----
print("\n--- Success Tier ---")
if "phase4" in all_summary:
    edl_agg = cv_summary.get("edl", {})
    if edl_agg and "acc" in edl_agg:
        edl_acc = edl_agg["acc"]["mean"]
        edl_f1 = edl_agg["macro_f1"]["mean"]
        if edl_acc >= 0.82 and edl_f1 >= 0.81:
            tier = "S -- Paper-ready SOTA!"
        elif edl_acc >= 0.80 and edl_f1 >= 0.79:
            tier = "A -- Beats all competitors!"
        elif edl_acc >= 0.78 and edl_f1 >= 0.77:
            tier = "B -- Competitive"
        else:
            tier = "C -- Investigate"
        print(f"  EDL CV: Acc={edl_acc:.4f} MacroF1={edl_f1:.4f} -> Tier: {tier}")

# ---- Save master ----
with open(os.path.join(OUTPUT_DIR, "v5_master_summary.json"), "w") as f:
    def convert_v5(obj):
        if isinstance(obj, dict):
            return {k: convert_v5(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_v5(v) for v in obj]
        elif isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    json.dump(convert_v5(all_summary), f, indent=2)

print("\n=== V5 EXPERIMENTS COMPLETE ===")
print("\nKey output files:")
for f in ["v5_phase1_results.json", "v5_phase2_kl_sweep.json",
          "v5_phase3_orcu_sweep.json", "v5_phase4_cv.json",
          "v5_master_summary.json"]:
    print(f"  {f}")
print("\nDownload /kaggle/working/ for all checkpoints, predictions, and result files.")
